In [2]:
import pandas as pd
import numpy as np
import os
import json 

os.getcwd()

'/Users/jillmarley/Scorecard-Metrics-Over-Time'

## Data
- release_data is the output from collecting release history (version publication dates) from GitHub REST API (see time_varying_covariates\release_history.ipynb) \\
- bq_data is the output from the inclusion / exclusion criteria applied in BQ

In [ ]:
release_data = pd.read_csv("full_release_history_final.csv")
bq_data = pd.read_csv("longitudinal_study_package_criteria_october.csv")

In [ ]:
## add a ProjectName column based on the url column
release_data['ProjectName'] = release_data['url'].str.extract(r'github\.com/([^/]+/[^/]+)')
release_data.rename(columns={'url': 'github_repo'}, inplace=True)

## add a github_repo
bq_data['github_repo'] = 'https://github.com/' + bq_data['ProjectName']

## **Step 4**

In [ ]:
## How many npm and PyPI packages are in Pat's dataset?
bq_npm = bq_data[bq_data['System'] == 'NPM']
bq_pypi = bq_data[bq_data['System'] == 'PYPI'] 
print(f"Number of npm packages in Pat's dataset: {len(bq_npm)}")
print(f"Number of PyPI packages in Pat's dataset: {len(bq_pypi)}")

Number of npm packages in Pat's dataset: 143186
Number of PyPI packages in Pat's dataset: 24523


In [ ]:
## need to map the package names in data to the urls in df_combined_final

df_combined_final = release_data.merge(
    bq_data[['ProjectName', 'Name', 'System']], 
    on='ProjectName', 
    how='right'
)

df_combined_final = df_combined_final.rename(columns={'ProjectName': 'project_name', 'Name': 'package_name'})

## remove columns
df_result = df_combined_final.drop(columns = ['release_name', 'target_commitish', 'repo', 'owner'])

## **Step 5**

In [7]:
## how many of the df_result repositories have status 'repo_not_found'?
repo_not_found_count = df_result[df_result['status'] == 'repo_not_found']['github_repo'].nunique()
print(f"Number of repositories with status 'repo_not_found': {repo_not_found_count}")

## print the number of PyPI and npm packages with 'repo_not_found' status
npm_repo_not_found = df_result[(df_result['System'] == 'NPM') & (df_result['status'] == 'repo_not_found')]['package_name'].nunique()
pypi_repo_not_found = df_result[(df_result['System'] == 'PYPI') & (df_result['status'] == 'repo_not_found')]['package_name'].nunique()
print(f"Number of npm packages with 'repo_not_found' status: {npm_repo_not_found}") 
print(f"Number of PyPI packages with 'repo_not_found' status: {pypi_repo_not_found}")


## print the number of npm packages after removing 'repo_not_found' status
npm_total = df_result[df_result['System'] == 'NPM']['package_name'].nunique()
npm_after_removal = npm_total - npm_repo_not_found
print(f"Number of npm packages after removing 'repo_not_found' status: {npm_after_removal}")

## print the number of PyPI packages after removing 'repo_not_found' status
pypi_total = df_result[df_result['System'] == 'PYPI']['package_name'].nunique()
pypi_after_removal = pypi_total - pypi_repo_not_found
print(f"Number of PyPI packages after removing 'repo_not_found' status: {pypi_after_removal}")

Number of repositories with status 'repo_not_found': 19754
Number of npm packages with 'repo_not_found' status: 18257
Number of PyPI packages with 'repo_not_found' status: 1212
Number of npm packages after removing 'repo_not_found' status: 121038
Number of PyPI packages after removing 'repo_not_found' status: 22657


## **Step 6: how many packages have release-level data?**
How many packages have release tag name equal to the tag name?

In [ ]:
## how many repositories have releases?
repo_with_releases = df_result[df_result['release_tag_name'].notna()]

## have many repositories have tag_name == release_tag_name and is not na?
repo_with_matching_tags = df_result[df_result['tag_name'] == df_result['release_tag_name']]
repo_with_matching_tags = repo_with_matching_tags[repo_with_matching_tags['release_tag_name'].notna()] 

print(f"Number of repositories with releases: {repo_with_releases['github_repo'].nunique()}")
print(f"Number of repositories with matching tag_name and release_tag_name: {repo_with_matching_tags['github_repo'].nunique()}")
print(f"Number of repositories with where no tag_name and release_tag_name values are the equal: {repo_with_releases['github_repo'].nunique() - repo_with_matching_tags['github_repo'].nunique()}")

## print the number of npm and pypi packages with matching tags
npm_with_releases = repo_with_matching_tags[repo_with_matching_tags['System'] == 'NPM']
pypi_with_releases = repo_with_matching_tags[repo_with_matching_tags['System'] == 'PYPI']

print(f"Number of npm packages with matching tag_name and release_tag_name: {npm_with_releases['github_repo'].nunique()}")
print(f"Number of PyPI packages with matching tag_name and release_tag_name: {pypi_with_releases['github_repo'].nunique()}")

Number of repositories with releases: 53127
Number of repositories with matching tag_name and release_tag_name: 53127
Number of repositories with where no tag_name and release_tag_name values are the equal: 0
Number of npm packages with matching tag_name and release_tag_name: 38712
Number of PyPI packages with matching tag_name and release_tag_name: 14465


## **Step 7: packages with releases published between 10/6/23 - 10/6/25:**

In [9]:
repo_with_matching_tags['published_at'] = pd.to_datetime(repo_with_matching_tags['published_at'], errors='coerce')

start = pd.Timestamp('2023-10-06', tz='UTC')
end = pd.Timestamp('2025-10-06', tz='UTC')

repo_with_matching_tags = repo_with_matching_tags[
    (repo_with_matching_tags['published_at'] >= start) &
    (repo_with_matching_tags['published_at'] <= end)
]

## print the number of unique repositories in repo_with_matching_tags after filtering by date
print(f"After filtering by date, there are {repo_with_matching_tags['package_name'].nunique()} unique packages where the release name matches the tag name and the published date is within one year between 2023-10-06 and 2025-10-06.")

## print the number of pypi vs npm packages after filtering by date
npm_count_date = repo_with_matching_tags[repo_with_matching_tags['System'] == 'NPM']['package_name'].nunique()
pypi_count_date = repo_with_matching_tags[repo_with_matching_tags['System'] == 'PYPI']['package_name'].nunique()
print(f"After filtering by date, number of unique npm packages: {npm_count_date}")
print(f"After filtering by date, number of unique PyPI packages: {pypi_count_date}")

After filtering by date, there are 19759 unique packages where the release name matches the tag name and the published date is within one year between 2023-10-06 and 2025-10-06.
After filtering by date, number of unique npm packages: 11503
After filtering by date, number of unique PyPI packages: 8316


## **Step 8: remove any packages with only one release between 10/6/23 - 10/6/25**

In [11]:
## need to determine how many repositories only have one row of data
repo_counts = repo_with_matching_tags['github_repo'].value_counts()
single_row_repos = repo_counts[repo_counts == 1]
print(f"Repositories with only one row of data: {len(single_row_repos)}")

## removing the repositories with only one row of data
repos_to_keep = repo_counts[repo_counts > 1].index
df_final_filtered_two_years = repo_with_matching_tags[repo_with_matching_tags['github_repo'].isin(repos_to_keep)]
print(f"Number of repositories after filtering: {df_final_filtered_two_years['github_repo'].nunique()}")
print(f"Number of packages after filtering: {df_final_filtered_two_years['package_name'].nunique()}")

## print the number of packages where system is 'npm' vs the number of packages where system is 'pypi'
npm_count = df_final_filtered_two_years[df_final_filtered_two_years['System'] == 'NPM']['package_name'].nunique()
pypi_count = df_final_filtered_two_years[df_final_filtered_two_years['System'] == 'PYPI']['package_name'].nunique()
print(f"Number of unique npm packages: {npm_count}")
print(f"Number of unique PyPI packages: {pypi_count}")

Repositories with only one row of data: 4181
Number of repositories after filtering: 16564
Number of packages after filtering: 15751
Number of unique npm packages: 9035
Number of unique PyPI packages: 6764


## **Step 9: remove repositories mapping to multiple packages**

In [13]:
# Keep only the first repository for each package (alphabetically)
data_deduped = df_final_filtered_two_years.sort_values(['package_name', 'github_repo']).drop_duplicates(subset='package_name', keep='first')
print(f"Only keeping the first repository for each package: {data_deduped['package_name'].nunique()} unique packages.")
print(f"Only keeping the first repository for each package: {data_deduped['project_name'].nunique()} unique projects.") 

print(f"There are {data_deduped['package_name'].nunique() - data_deduped['project_name'].nunique()} projects mapping to multiple packages.") 

# Now we end up with more packages than projects. We need to delete all packages and projects where a project maps to multiple packages
projects_multiple_packages = data_deduped['project_name'][data_deduped['project_name'].duplicated(keep=False)].unique()
data_deduped_v2 = data_deduped[~data_deduped['project_name'].isin(projects_multiple_packages)]

## now remove the rows in our original dataframe, data, that are not found in data_deduped_v2 keeping all of the columsn in the original dataframe
data_final = df_final_filtered_two_years[df_final_filtered_two_years['package_name'].isin(data_deduped_v2['package_name']) & df_final_filtered_two_years['project_name'].isin(data_deduped_v2['project_name'])]

Only keeping the first repository for each package: 15751 unique packages.
Only keeping the first repository for each package: 15727 unique projects.
There are 24 projects mapping to multiple packages.


In [14]:
## print the number of pypi and npm packages in data_final
npm_final_count = data_final[data_final['System'] == 'NPM']['package_name'].nunique()
pypi_final_count = data_final[data_final['System'] == 'PYPI']['package_name'].nunique()
print(f"Final number of unique npm packages: {npm_final_count}")
print(f"Final number of unique PyPI packages: {pypi_final_count}")

Final number of unique npm packages: 9004
Final number of unique PyPI packages: 6711


In [16]:
## are any packages in both ecosystems?
npm_packages = set(data_final[data_final['System'] == 'NPM']['package_name'].unique())
pypi_packages = set(data_final[data_final['System'] == 'PYPI']['package_name'].unique())
overlap_packages = npm_packages.intersection(pypi_packages)
print(f"Number of packages in both ecosystems: {len(overlap_packages)}")
## print the overlapping packages and corresponding github repositories
for package in overlap_packages:
    npm_repos = data_final[(data_final['System'] == 'NPM') & (data_final['package_name'] == package)]['github_repo'].unique()
    pypi_repos = data_final[(data_final['System'] == 'PYPI') & (data_final['package_name'] == package)]['github_repo'].unique()
    print(f"Package: {package}")
    print(f"  NPM Repositories: {', '.join(npm_repos)}")
    print(f"  PyPI Repositories: {', '.join(pypi_repos)}")

Number of packages in both ecosystems: 12
Package: bqplot
  NPM Repositories: https://github.com/bloomberg/bqplot
  PyPI Repositories: https://github.com/bloomberg/bqplot
Package: jupyterlab-pioneer
  NPM Repositories: https://github.com/educational-technology-collective/jupyterlab-pioneer
  PyPI Repositories: https://github.com/educational-technology-collective/jupyterlab-pioneer
Package: wslink
  NPM Repositories: https://github.com/kitware/wslink
  PyPI Repositories: https://github.com/kitware/wslink
Package: vosk
  NPM Repositories: https://github.com/alphacep/vosk-api
  PyPI Repositories: https://github.com/alphacep/vosk-api
Package: jupyterlab-tour
  NPM Repositories: https://github.com/jupyterlab-contrib/jupyterlab-tour
  PyPI Repositories: https://github.com/jupyterlab-contrib/jupyterlab-tour
Package: tokenizers
  NPM Repositories: https://github.com/huggingface/tokenizers
  PyPI Repositories: https://github.com/huggingface/tokenizers
Package: langsmith
  NPM Repositories: http

## **Step 10: How many repositories have unique GitHub IDs?** 
GitHub automatically redirects when repository name changes, but the previous names of repositories remain in deps.dev.

In [17]:
import pandas as pd

data = pd.read_csv('package_data.csv')
repo_list = data['githubrepo'].unique().tolist()

In [18]:
repo_ids = pd.read_csv('repo_ids_mapping.csv')

In [19]:
## add a repo_ids column to the data based on repo_name_at_some_point and project_name
data = data.merge(repo_ids, left_on='project_name', right_on='repo_name_at_some_point', how='left')

In [21]:
## print the number of unique IDs corresponding to npm and PyPI packages
npm_ids = data[data['System'] == 'NPM']['repo_id'].nunique()
pypi_ids = data[data['System'] == 'PYPI']['repo_id'].nunique()
print(f"Number of unique repo IDs for npm packages: {npm_ids}")
print(f"Number of unique repo IDs for PyPI packages: {pypi_ids}")

## print the total number of unique IDs 
total_ids = data['repo_id'].nunique()
print(f"Total number of unique repo IDs: {total_ids}")

Number of unique repo IDs for npm packages: 8721
Number of unique repo IDs for PyPI packages: 6634
Total number of unique repo IDs: 15340


In [22]:
npm_set = set(data[data['System'] == 'NPM']['repo_id'])
pypi_set = set(data[data['System'] == 'PYPI']['repo_id'])

overlap = npm_set.intersection(pypi_set)
print(f"Number of overlapping IDs: {len(overlap)}")
print(overlap)

Number of overlapping IDs: 15
{43191201.0, 610119522.0, 90787393.0, 91253698.0, 647427819.0, 171410703.0, 357544887.0, 273939379.0, 206138137.0, 690723511.0, 219035799.0, 186670393.0, 194738138.0, 370931483.0, 262943932.0}
